In [13]:
!pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic
!pip install "crewai[azure-ai-inference]"
%pip install "litellm==1.105.1"
!pip install crewai_tools# Import the libraries , all of the tools for agent building
!pip install langchain openai PyPDF2 faiss-cpu tiktoken langchain-openai langchain-classic pypdf
!pip install langchain_community
!pip install langchain_text_splitters
!pip install langchain_classic

Note: you may need to restart the kernel to use updated packages.
ERROR: Invalid requirement: 'crewai_tools#'


In [14]:
!pip install crewai_tools

In [15]:
import os
from datetime import datetime, timezone
import requests
from crewai.llm import LLM
from crewai import Agent, Task, Crew, Process
from crewai.tools import BaseTool
from crewai_tools import PDFSearchTool

from crewai_tools import PDFSearchTool
from crewai_tools.rag.data_types import DataType
from crewai_tools.rag.loaders.pdf_loader import PDFLoader
from crewai_tools.rag.base_loader import SourceContent
from crewai_tools.rag.chunkers.text_chunker import TextChunker


In [16]:
# Use the EXACT deployment names created in your Azure OpenAI resource.
# If the deployment name is different from the underlying model name, keep the deployment name here.
AZURE_OPENAI_API_KEY = "2ABecnfxzhRg4M5D6pBKiqxXVhmGB2WvQ0aYKkbTCPsj0JLKsZPfJQQJ99BDAC77bzfXJ3w3AAABACOGi3sC"
AZURE_OPENAI_ENDPOINT = "https://openai-api-management-gw.azure-api.net/"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"
AZURE_OPENAI_DEPLOYMENT = "gpt-5-mini"  # must match the Azure deployment name you created
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"  # must match the Azure embedding deployment name you created


In [17]:
# Tavily API Key
TAVILY_API_KEY = "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"#"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"

In [18]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "Web Search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"

        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()
        results = data.get("results", [])

        if not results:
            return "No search results returned."

        formatted_results = []
        for r in results:
            formatted_results.append(f"{r.get('title', 'Untitled')} - {r.get('url', '')}")

        return "\n".join(formatted_results)


search_tool = TavilySearchTool()


In [19]:
# 1. Test Tavily API Connection
try:
    print("Testing Tavily API...")
    test_search = search_tool._run("AI healthcare advancements 2024")
    print("Tavily API is working perfectly!")
    print("Sample Output:\n", "\n".join(test_search.split("\n")[:2]))
except Exception as e:
    print(f"Tavily API Error: {e}")

Testing Tavily API...
Tavily API is working perfectly!
Sample Output:
 Artificial Intelligence in Healthcare: 2024 Year in Review - https://www.medrxiv.org/content/10.1101/2025.02.26.25322978v1.full
IEEE SA - Five Healthcare and Life Sciences Trends to Watch for in 2024 - https://standards.ieee.org/beyond-standards/2024-healthcare-and-life-sciences-trends


In [20]:
# --------------------------------------------------
# Azure GPT-5-mini LLM via CrewAI LiteLLM path
# --------------------------------------------------
llm = LLM(
    model="azure/gpt-5-mini",
    api_key=AZURE_OPENAI_API_KEY,
    base_url=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    is_litellm=True,
    temperature=0,
    max_completion_tokens=1500
)

In [21]:
# 2. Test Azure OpenAI Connection
try:
    print("Testing Azure OpenAI connection...")
    test_response = llm.call(messages=[{"role": "user", "content": "Respond with 'Azure OpenAI connection successful!'"}])
    print(test_response)
except Exception as e:
    print(f"Azure OpenAI Error: {e}")

Testing Azure OpenAI connection...
Azure OpenAI connection successful!


In [35]:
import importlib.metadata as md

for p in ["crewai", "crewai-tools", "litellm", "mcp", "tavily-python"]:
    try:
        print(p, md.version(p))
    except md.PackageNotFoundError:
        print(p, "not installed")

crewai 1.15.30
crewai-tools 1.15.30
litellm 1.105.1
mcp 1.30.0
tavily-python 0.8.5


Test pdf search tool of crewai_tools

PDF search tool for static, domain-specific content 

Defaults to using OpenAI's API for embeddings and search, which requires the OPENAI_API_KEY environment variable to be set. Since you are using Azure OpenAI, we can pass our custom Azure LLM configuration directly into the tool config so it does not look for a standard OPENAI_API_KEY.

We need to add controlled chunk into the same vector database, without relying on unsupported config keys. Issue is that for bigger pdf documents , the chunking strategy fails as PDFSearchTool does not publicly expose any custom chunker size and tries to load the entire document at once causing RateLimitError.Hence enforce a manually crafter chunker to split the pdf first and add it iteratively into the vector store . 

In [23]:
# pip install pypdf chromadb openai crewai
import time
import chromadb
from pypdf import PdfReader
from openai import AzureOpenAI, RateLimitError
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

PDF_PATH = "/home/subhroy468/router-retriever/venv/attentionPaper.pdf"
CHROMA_DIR = "./chroma_store"
COLLECTION = "attention_paper"
BATCH_SIZE = 16  # stays under Azure's per-request input limit

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# ---------- 1) Chunk, page by page ----------
def chunk_text(text, size=1000, overlap=150):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks

def load_chunks(pdf_path):
    reader = PdfReader(pdf_path)
    docs, metas, ids = [], [], []
    for page_no, page in enumerate(reader.pages, start=1):
        text = (page.extract_text() or "").strip()
        for i, chunk in enumerate(chunk_text(text)):
            if chunk.strip():
                docs.append(chunk)
                metas.append({"page": page_no})
                ids.append(f"p{page_no}-c{i}")
    return docs, metas, ids

# ---------- 2) Embed in small batches ----------
def embed_batch(texts, retries=5):
    for attempt in range(retries):
        try:
            resp = client.embeddings.create(
                model=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,  # deployment name
                input=texts,
            )
            return [d.embedding for d in resp.data]
        except RateLimitError:
            time.sleep(2 ** attempt)
    raise RuntimeError("Embedding failed after retries")

# ---------- 3) Store in Chroma (persistent, skip if already built) ----------
chroma = chromadb.PersistentClient(path=CHROMA_DIR)
collection = chroma.get_or_create_collection(COLLECTION)

def build_index():
    if collection.count() > 0:
        print(f"Index exists with {collection.count()} chunks, skipping.")
        return
    docs, metas, ids = load_chunks(PDF_PATH)
    print(f"{len(docs)} chunks to embed")
    for i in range(0, len(docs), BATCH_SIZE):
        d, m, idx = docs[i:i+BATCH_SIZE], metas[i:i+BATCH_SIZE], ids[i:i+BATCH_SIZE]
        collection.add(documents=d, metadatas=m, ids=idx, embeddings=embed_batch(d))
        print(f"  embedded {min(i+BATCH_SIZE, len(docs))}/{len(docs)}")

build_index()

# ---------- Custom tool for the agent ----------
@tool("Search Attention Paper")
def search_paper(query: str) -> str:
    """Semantic search over the 'Attention Is All You Need' PDF. Returns the most relevant passages with page numbers."""
    q_emb = embed_batch([query])[0]
    res = collection.query(query_embeddings=[q_emb], n_results=4)
    return "\n\n".join(
        f"[page {m['page']}] {doc}"
        for doc, m in zip(res["documents"][0], res["metadatas"][0])
    )


Index exists with 43 chunks, skipping.


In [24]:
# ---------- 4) Test the tool ----------
test_query = "Which limitation in transduction does the paper discuss?"
print("Query:", test_query)
response = search_paper._run(test_query)
print("Result:\n", response)


Query: Which limitation in transduction does the paper discuss?
Result:
 [page 9] 3 26.4 213
In Table 3 rows (B), we observe that reducing the attention key size dk hurts model quality. This
suggests that determining compatibility is not easy and that a more sophisticated compatibility
function than dot product may be beneﬁcial. We further observe in rows (C) and (D) that, as expected,
bigger models are better, and dropout is very helpful in avoiding over-ﬁtting. In row (E) we replace our
sinusoidal positional encoding with learned positional embeddings [8], and observe nearly identical
results to the base model.
7 Conclusion
In this work, we presented the Transformer, the ﬁrst sequence transduction model based entirely on
attention, replacing the recurrent layers most commonly used in encoder-decoder architectures with
multi-headed self-attention.
For translation tasks, the Transformer can be trained signiﬁcantly faster than architectures based
on recurrent or convolutional layers. On

**Quick test to PDF retrieval by creating a crew with a single agent and enable tracing and vberbose for reviewing the reasonaing steps**

In [ ]:
import asyncio
agent = Agent(
    role="Paper Q&A Assistant",
    goal="Answer questions about the paper using the retrieval tool.",
    backstory="You answer questions about ML papers and cite page numbers.",
    llm=llm,
    tools=[search_paper],
    allow_delegation=False,
    verbose=True,   # temporarily, so you can see where it fails
)

task = Task(
    description="Use only the pdf document to find what limitation in transduction the paper discusses.",
    expected_output="A short answer with page references.",
    agent=agent,
)

crew = Crew(agents=[agent], tasks=[task], verbose=True)

try:
    result = await crew.kickoff_async()
    print(result.raw if hasattr(result, "raw") else result)
except Exception as e:
    print(type(e))
    print(getattr(e, "body", None))
    print(str(e)[:2000])


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 69c2def6-1c33-45b7-94f8-8294b2970b62                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use only the pdf document to find what limitation in transduction the paper discusses.                   │
│  ID: 31b6db92-310e-447a-98ba-8228386a5acd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Paper Q&A Assistant                                                                                     │
│                                                                                                                 │
│  Task: Use only the pdf document to find what limitation in transduction the paper discusses.                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#4) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_attention_paper                                                                                   │
│  Args: {'query': 'transduction limitation transduction limitation in transduction Attention Is All You Need     │
│  transduction section'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#4) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: search_attention_paper                                                                                   │
│  Output: [page 1] Attention Is All You Need                                                                     │
│  Ashish Vaswani∗                                                                                                │
│  Google Brain                                                                                                   │
│  avaswani@google.com                                                                                            │
│  Noam Shazeer∗                                                                                                  │
│  Google Brain                                                                                                   │
│  noam@google.com                                                                                                │
│  Niki Parmar∗                                                                                                   │
│  Google Research                                                                                                │
│  nikip@google.com                                                                                               │
│  Jakob Uszkoreit∗                                                                                               │
│  Google Research                                                                                                │
│  usz@google.com                                                                                                 │
│  Llion Jones∗                                                                                                   │
│  Google Research                                                                                                │
│  llion@google.com                                                                                               │
│  Aidan N. Gomez∗†                                                                                               │
│  University of Toronto                                                                                          │
│  aidan@cs.toronto.edu                                                                                           │
│  Łukasz Kaiser∗                                                                                                 │
│  Google Brain                                                                                                   │
│  lukaszkaiser@google.com                                                                                        │
│  Illia Polosukhin∗‡                                                                                             │
│  illia.polosukhin@gmail.com                                                                                     │
│  Abstract                                                                                                       │
│  The dominant sequence transduction models are based on complex recurrent or                                    │
│  convolutional neural networks that include an encoder and a decoder. The best                                  │
│  performing models also connect the encoder and decoder through an attention                                    │
│  mechanism. We propose a new simple network architecture, the Transformer,                                      │
│  based solely on attention mechanisms, dispensing with recurrence and convolutions                              │
│  entirely. Experiments on two machine translation tasks

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Paper Q&A Assistant                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  "Recurrent models typically factor computation along the symbol positions of the input and output sequences.   │
│  Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a        │
│  function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature     │
│  precludes parallelization within training examples, which becomes critical at longer sequence lengths, as      │
│  memory constraints limit batching across examples. Recent work has achieved significant improvements in        │
│  computational efficiency through factorization tricks [18] and conditional computation [26], while also        │
│  improving model performance in case of the latter. The fundamental constraint of sequential computation,       │
│  however, remains." (page 2)                                                                                    │
│                                                                                                                 │
│  "In these models, the number of operations required to relate signals from two arbitrary input or output       │
│  positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This  │
│  makes it more difficult to learn dependencies between distant positions [11]." (page 2)                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Use only the pdf document to find what limitation in transduction the paper discusses.                   │
│  Agent: Paper Q&A Assistant                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

"Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [18] and conditional computation [26], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains." (page 2)

"In these models, the number of operations required to relate signals from two arbitrary input or output positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes it more dif

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 69c2def6-1c33-45b7-94f8-8294b2970b62                                                                       │
│  Final Output: "Recurrent models typically factor computation along the symbol positions of the input and       │
│  output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden      │
│  states ht, as a function of the previous hidden state ht−1 and the input for position t. This inherently       │
│  sequential nature precludes parallelization within training examples, which becomes critical at longer         │
│  sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant   │
│  improvements in computational efficiency through factorization tricks [18] and conditional computation [26],   │
│  while also improving model performance in case of the latter. The fundamental constraint of sequential         │
│  computation, however, remains." (page 2)                                                                       │
│                                                                                                                 │
│  "In these models, the number of operations required to relate signals from two arbitrary input or output       │
│  positions grows in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This  │
│  makes it more difficult to learn dependencies between distant positions [11]." (page 2)                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Do a quick test for the retrieval to test that it answers questions grounded to the queries 
query1 -  Provide an irrelevant question 
query2 - Specific questions asking for names of authors of the research paper . 

In [26]:
query1 = "How can food be manufactured in Mars planet?"

passages = search_paper._run(query1)
#print(passages)

# Optional: only if the deployment accepts the prompt
answer = llm.call(
    messages=[
        {
            "role": "user",
            "content": (
                "Use only the following passages to answer the question. "
                "If the passage does not clearly answer, say 'Not enough evidence in the provided text.'\n\n"
                f"{passages}\n\nQuestion: {query1}"
            )
        }
    ]
)
print("query1:", query1)
print(answer)

#  Repeat the above for query2
query2 = "Simply name the authors of the given research paper and provide which year was this published?"
passages2 = search_paper._run(query2)
answer2 = llm.call(
    messages=[
        {
            "role": "user",
            "content": (
                "Use only the following passages to answer the question. "
                "If the passage does not clearly answer, say 'Not enough evidence in the provided text.'\n\n"
                f"{passages2}\n\nQuestion: {query2}"
            )
        }
    ]
)
print("query2:", query2)
print(answer2)

query1: How can food be manufactured in Mars planet?
Not enough evidence in the provided text.
query2: Simply name the authors of the given research paper and provide which year was this published?
Jakob; Ashish; Illia; Noam; Niki; Llion; Lukasz; Aidan — 2017


Router Agent classifies the question for the router to decide on the retrieval using direct LLM vs RAG vs web search tool.
Retriever Agent executes the most suitable retrieval method, using either a PDF-based vector search or a web search tool. 
The system demonstrates intelligent orchestration across these agents, producing accurate, source-grounded answers.

PDF search: Retrieve information from static, domain-specific documents.

Web search: Retrieve fresh information from the internet using Tavily search tool.

Direct LLM generation: An optional path that answers without retrieval where question unrelated to document or web search requirement is asked for.

Router decision is fed into the retriever agent , one among the following - 

WEB_SEARCH

DIRECT_LLM

PDF_SEARCH

In [ ]:
# Explicit prompt for the router input
print("\n=== Router + Retriever Demo ===")
user_query = input("Enter your question for the router agent: ").strip()
if not user_query:
    user_query = "What are the authors of this paper and what year was it published?"

print(f"\nRouting this question: {user_query}\n")

# Define the router agent
routerAgent = Agent(
    role="Analyzes and classifies the question",
    goal="Decide whether to use PDF search, web search, or direct LLM",
    backstory="You decide the retrieval path based on the user question.",
    llm=llm,
    allow_delegation=False,
    verbose=True,
)

# Define the retriever agent
retrieverAgent = Agent(
    role="Generates the answer by choosing the appropriate tool based on the routing decision",
    goal="Use the selected retrieval path to answer the question",
    backstory="You generate answers based on the routing decision made by the router agent.",
    llm=llm,
    tools=[search_paper, search_tool],
    allow_delegation=False,
    verbose=True,
)

routerTask = Task(
    description=(
        f"Classify this user question and return only one route: "
        f"PDF_SEARCH, WEB_SEARCH, or DIRECT_LLM. "
        f"User question: {user_query}"
    ),
    expected_output="Exactly one of: PDF_SEARCH, WEB_SEARCH, DIRECT_LLM",
    agent=routerAgent,
)

retrieverTask = Task(
    description=(
        f"Use the routing decision from the router to answer the user's question. "
        f"User question: {user_query}"
    ),
    expected_output="A final answer grounded in the chosen retrieval method.",
    agent=retrieverAgent,
    context=[routerTask],
)

crewrr = Crew(
    agents=[routerAgent, retrieverAgent],
    tasks=[routerTask, retrieverTask],
    process=Process.sequential,
    verbose=True,
)

try:
    answer = await crewrr.kickoff_async()
    print(answer.raw if hasattr(answer, "raw") else answer)
except Exception as e:
    print(type(e))
    print(getattr(e, "body", None))
    print(str(e)[:2000])



=== Router + Retriever Demo ===

Routing this question: What are most recent advancements in Jev in 2026 and what are its applications?



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2631a31d-49cc-434f-90c2-5f814271d371                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Classify this user question and return only one route: PDF_SEARCH, WEB_SEARCH, or DIRECT_LLM. User       │
│  question: What are most recent advancements in Jev in 2026 and what are its applications?                      │
│  ID: 085e042d-f53e-4686-9c8d-d72873d3bef0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analyzes and classifies the question                                                                    │
│                                                                                                                 │
│  Task: Classify this user question and return only one route: PDF_SEARCH, WEB_SEARCH, or DIRECT_LLM. User       │
│  question: What are most recent advancements in Jev in 2026 and what are its applications?                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭─────────────────────────────────────── ✅ Tool Execution Completed (#18) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: web_search                                                                                               │
│  Output: The future of Japanese encephalitis vaccination: expert recommendations for achieving and maintaining  │
│  optimal JE control | npj Vaccines - https://www.nature.com/articles/s41541-021-00338-z                         │
│  Japanese Encephalitis Virus: An Update on the Potential Antivirals and Vaccines -                              │
│  https://pmc.ncbi.nlm.nih.gov/articles/PMC10146181                                                              │
│  Japanese encephalitis - https://www.who.int/news-room/fact-sheets/detail/japanese-encephalitis                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#19) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': "PMC10146181 'Japanese Encephalitis Virus: An Update on the Potential Antivirals and           │
│  Vaccines' abstract PMC10146181 2022"}                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#19) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: web_search                                                                                               │
│  Output: Japanese Encephalitis Virus: An Update on the Potential Antivirals and Vaccines -                      │
│  https://pmc.ncbi.nlm.nih.gov/articles/PMC10146181                                                              │
│  Recent Advances in Antivirals for Japanese Encephalitis Virus - https://www.mdpi.com/1999-4915/15/5/1033       │
│  Frontiers | Partial protective efficacy of the current licensed Japanese encephalitis live vaccine against     │
│  the emerging genotype I Japanese encephalitis virus isolated from sheep -                                      │
│  https://www.frontiersin.org/journals/immunology/articles/10.3389/fimmu.2025.1513261/full                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#20) ────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {'query': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC10146181 full text Japanese Encephalitis Virus: An   │
│  Update on the Potential Antivirals and Vaccines'}                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────── ✅ Tool Execution Completed (#20) ───────────────────────────────────────╮
│                                                                                                                 │
│  Tool Completed                                                                                                 │
│  Tool: web_search                                                                                               │
│  Output: Japanese Encephalitis Virus: An Update on the Potential Antivirals and Vaccines -                      │
│  https://www.mdpi.com/2076-393X/11/4/742                                                                        │
│  Japanese Encephalitis Virus: An Update on the Potential Antivirals and Vaccines -                              │
│  https://pmc.ncbi.nlm.nih.gov/articles/PMC10146181                                                              │
│  [PDF] Japanese Encephalitis Virus: An Update on the Potential Antivirals and Vaccines | Semantic Scholar -     │
│  https://www.semanticscholar.org/paper/Japanese-Encephalitis-Virus%3A-An-Update-on-the-and-Srivastava-Jeswani/  │
│  66cb69d52fcded37bd52ff6b41bb863257178345                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**SEPARATOR**